# Efficient Frontier & Estimation-Error Walkthrough

Runs the estimation and optimization layers on the cached data and visualizes the frontier.

**Run after `python scripts/download_data.py`.**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve() / 'src'))

import matplotlib.pyplot as plt

from portfolio_optimization_lab.config import DEFAULT_CONFIG
from portfolio_optimization_lab.data import load_price_frame, log_returns
from portfolio_optimization_lab.estimates import estimate, portfolio_stats
from portfolio_optimization_lab.optimization import efficient_frontier, min_variance, max_sharpe, risk_parity

## 1. Estimates on the full sample

In [ ]:
cfg = DEFAULT_CONFIG
prices = load_price_frame(cfg.tickers, cfg.benchmark, cfg.raw_dir)
rets = log_returns(prices[list(cfg.tickers)])
est = estimate(rets, cfg.periods_per_year)
print('Ledoit-Wolf shrinkage intensity:', round(est.shrinkage, 3))
print(est.mu.round(3))

## 2. The constrained efficient frontier

In [ ]:
frontier = efficient_frontier(est.mu, est.cov, max_weight=cfg.max_weight, n_points=40)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(frontier['volatility'], frontier['target_return'], lw=2)
for name, w in {'min var': min_variance(est.mu, est.cov, cfg.max_weight),
                'max sharpe': max_sharpe(est.mu, est.cov, cfg.rf_annual, cfg.max_weight),
                'risk parity': risk_parity(est.cov)}.items():
    s = portfolio_stats(w, est.mu, est.cov)
    ax.scatter(s['volatility'], s['return'], marker='*', s=120)
    ax.annotate(name, (s['volatility'], s['return']), xytext=(6, -3), textcoords='offset points')
ax.set_xlabel('annualized volatility'); ax.set_ylabel('annualized return'); ax.set_title('Constrained efficient frontier')
fig.tight_layout(); plt.show()

## Takeaway

The frontier looks beautiful in-sample. The research question — and the walk-forward backtest in `scripts/run_optimization.py` — is how much of it survives out of sample.